In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import pandas as pd
import numpy as np

import os
import pandas as pd
import numpy as np

#from keras.layers.wrappers import Bidirectional

In [5]:
data=pd.read_csv('/content/drive/MyDrive/HMPV Sentiment analysis/sentiment_output (1).csv')[['Cleaned_Translated_Comment','Sentiment_Label']]
data.dropna()
data

,Cleaned_Translated_Comment,Sentiment_Label
0,apparently its been around since 2000 so nothi...,Neutral
1,everyone its just a cold me back to the bunker,Neutral
2,have had hmpv here in australia since before c...,Negative
3,iâtrade_markve got this right now and have bee...,Negative
4,2020 was a wtf year and there was covid 2025 i...,Negative
...,...,...
9753,the virus is already on the way to india flyin...,Positive
9754,why only this news about various,Neutral
9755,another cover up china should pay the world fo...,Negative
9756,were not bying it go away,Neutral


In [6]:
character_set_not =  ['\n', '!', '"', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', ':', ';', '=', '?',
                      '[', '\\', ']', '{', '}', '\xa0', '¬', '´', '·', '।', '\u09e4', '৷', '\u200d', '–', '—', '‘', '’', '“', '”', '•']

In [7]:
def tokenize(s):
    for i in character_set_not:
        s =  s.replace(i, '')
    s=s.lower()
    return  s

In [8]:
'''train['Text']=train['Text'].map(tokenize)
train'''

"train['Text']=train['Text'].map(tokenize)\ntrain"

In [9]:
data['Cleaned_Translated_Comment']=data['Cleaned_Translated_Comment'].map(tokenize)
data

,Cleaned_Translated_Comment,Sentiment_Label
0,apparently its been around since 2000 so nothi...,Neutral
1,everyone its just a cold me back to the bunker,Neutral
2,have had hmpv here in australia since before c...,Negative
3,iâtrade_markve got this right now and have bee...,Negative
4,2020 was a wtf year and there was covid 2025 i...,Negative
...,...,...
9753,the virus is already on the way to india flyin...,Positive
9754,why only this news about various,Neutral
9755,another cover up china should pay the world fo...,Negative
9756,were not bying it go away,Neutral


In [10]:
labels = [0,1,2]
labels

[0, 1, 2]

In [11]:
data.columns

Index(['Cleaned_Translated_Comment', 'Sentiment_Label'], dtype='object')

In [12]:
import numpy as np
from sklearn.model_selection import train_test_split

# Load your dataset and split it into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(data['Cleaned_Translated_Comment'],data['Sentiment_Label'], test_size=0.2, random_state=42)

'''X_train= train['Text']
X_val= test['Text']
y_val= test['Relation']
y_train= train['Relation']'''

"X_train= train['Text']\nX_val= test['Text']\ny_val= test['Relation']\ny_train= train['Relation']"

In [13]:
from transformers import XLNetTokenizerFast, TFXLNetForSequenceClassification
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import tensorflow as tf
import numpy as np

# Load XLNet Tokenizer
tokenizer = XLNetTokenizerFast.from_pretrained("xlnet-base-cased")

# Tokenizing
train_encodings = tokenizer(list(X_train), truncation=True, padding=True, max_length=512, return_tensors="tf")
test_encodings = tokenizer(list(X_val), truncation=True, padding=True, max_length=512, return_tensors="tf")

# Label Encoding
label_encoder = LabelEncoder()
label_encoder.fit(y_train)

# Encode the labels
y_train = label_encoder.transform(y_train)
y_test = label_encoder.transform(y_val)

# Display label mapping
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print("Label Encoder Mapping:")
for original_label, encoded_label in label_mapping.items():
    print(f"Original Label: {original_label} -> Encoded Label: {encoded_label}")

# Load XLNet Model for Sequence Classification
num_labels = len(label_mapping)
model = TFXLNetForSequenceClassification.from_pretrained("xlnet-base-cased", num_labels=num_labels)

# Optimizer, Loss, and Metrics
optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5, epsilon=1e-08, clipnorm=1.0)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metric = tf.keras.metrics.SparseCategoricalAccuracy("accuracy")

model.compile(optimizer=optimizer, loss=loss, metrics=[metric])
model.summary()

# Create TensorFlow Datasets
train_dataset = tf.data.Dataset.from_tensor_slices((dict(train_encodings), np.array(y_train)))
test_dataset = tf.data.Dataset.from_tensor_slices((dict(test_encodings), np.array(y_test)))

# Training the model
model.fit(
    train_dataset.batch(8),
    epochs=1,  # Adjust epochs as needed
    validation_data=test_dataset.batch(8)
)

# Predict on the test dataset
predictions = model.predict(test_dataset.batch(8))

# Get predicted class labels (from logits)
y_pred = tf.argmax(predictions.logits, axis=-1).numpy()

# Calculate Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')  # Weighted for multiclass
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

# Print metrics
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

Label Encoder Mapping:
Original Label: Negative -> Encoded Label: 0
Original Label: Neutral -> Encoded Label: 1
Original Label: Positive -> Encoded Label: 2


tf_model.h5:   0%|          | 0.00/565M [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/tf_keras/src/initializers/initializers.py:121: UserWarning: The initializer TruncatedNormal is unseeded and being called multiple times, which will return identical values each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initializer instance more than once.
  warnings.warn(
Some layers from the model checkpoint at xlnet-base-cased were not used when initializing TFXLNetForSequenceClassification: ['lm_loss']
- This IS expected if you are initializing TFXLNetForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFXLNetForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a Be

Model: "tfxl_net_for_sequence_classification"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 transformer (TFXLNetMainLa  multiple                  116718336 
 yer)                                                            
                                                                 
 sequence_summary (TFSequen  multiple                  590592    
 ceSummary)                                                      
                                                                 
 logits_proj (Dense)         multiple                  2307      
                                                                 
Total params: 117311235 (447.51 MB)
Trainable params: 117311235 (447.51 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


123/163 [=====================>........] - ETA: 12s

ResourceExhaustedError: Graph execution error:

Detected at node tfxl_net_for_sequence_classification/transformer/layer_._0/rel_attn/einsum_5/Einsum defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/usr/local/lib/python3.11/dist-packages/colab_kernel_launcher.py", line 37, in <module>

  File "/usr/local/lib/python3.11/dist-packages/traitlets/config/application.py", line 992, in launch_instance

  File "/usr/local/lib/python3.11/dist-packages/ipykernel/kernelapp.py", line 619, in start

  File "/usr/local/lib/python3.11/dist-packages/tornado/platform/asyncio.py", line 195, in start

  File "/usr/lib/python3.11/asyncio/base_events.py", line 608, in run_forever

  File "/usr/lib/python3.11/asyncio/base_events.py", line 1936, in _run_once

  File "/usr/lib/python3.11/asyncio/events.py", line 84, in _run

  File "/usr/local/lib/python3.11/dist-packages/tornado/ioloop.py", line 685, in <lam

In [14]:
from sklearn.metrics import confusion_matrix
import numpy as np

# Assuming you have your test_dataset and true labels ready
test_dataset = tf.data.Dataset.from_tensor_slices((dict(test_encodings), np.array(y_val))).batch(12)

# Get the true labels and ensure they are numerical using label encoding
y_true = label_encoder.transform(y_val) # Use label_encoder to transform y_val into numerical labels

# Make predictions on the test dataset
y_pred = model.predict(test_dataset).logits
y_pred = np.argmax(y_pred, axis=1)

# Calculate the confusion matrix
conf_matrix = confusion_matrix(y_true, y_pred)

print("Confusion Matrix:\n", conf_matrix)

ResourceExhaustedError: Graph execution error:

Detected at node tfxl_net_for_sequence_classification/transformer/layer_._0/rel_attn/einsum_5/Einsum defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/usr/local/lib/python3.11/dist-packages/colab_kernel_launcher.py", line 37, in <module>

  File "/usr/local/lib/python3.11/dist-packages/traitlets/config/application.py", line 992, in launch_instance

  File "/usr/local/lib/python3.11/dist-packages/ipykernel/kernelapp.py", line 619, in start

  File "/usr/local/lib/python3.11/dist-packages/tornado/platform/asyncio.py", line 195, in start

  File "/usr/lib/python3.11/asyncio/base_events.py", line 608, in run_forever

  File "/usr/lib/python3.11/asyncio/base_events.py", line 1936, in _run_once

  File "/usr/lib/python3.11/asyncio/events.py", line 84, in _run

  File "/usr/local/lib/python3.11/dist-packages/tornado/ioloop.py", line 685, in <lam

In [ ]:
id2labels = model.config.id2label
model.config.id2label = {id : label_encoder.inverse_transform([id])[0]  for id, label in id2labels.items()}

label2ids = model.config.label2id
model.config.label2id = {label_encoder.inverse_transform([id])[0] : id   for id, label in id2labels.items()}

In [ ]:
!pip install shap

In [ ]:
import transformers
import shap

In [ ]:
pred = transformers.pipeline("text-classification", model=model, tokenizer=tokenizer, device=0, return_all_scores=True)
explainer = shap.Explainer(pred)

In [ ]:
ls=[
    "My parents are non believers when it comes with viruses so i dont know how i think of this",#Neu
    "Depending on your religion, and maybe even your intellect, metaphysically speaking, this might be one of those new viral pandemics being spread by Uranus, or by Jupiter or perhaps even Mars, or, god help us, Saturn ~ The god's must be going mad again...  (Nevermind, Trump will save America, and then America will save the rest of us)...", #Pos
    'This is not real news. The Hmpv virus is the common flu.  No proof of anything and no confirmation on the ground and they still report on it !!  This is just bad journalism, no one serious is reporting on this anywhere around the world â€¦.  This news is pure FEAR MONGERING' #Neg
]
ls=[tokenize(s) for s in ls]

In [ ]:
shap_values = explainer(ls)

In [ ]:
classes=['Negative','Neutral','Positive']

In [ ]:
shap.plots.text(shap_values, display=True)

In [ ]:
import matplotlib.pyplot as plt
from IPython.core.display import HTML
HTML(shap.plots.text(shap_values, display=True))
with open('/content/drive/MyDrive/HMPV Sentiment analysis/Figures/shapfinal.html', 'w') as file:
     file.write(shap.plots.text(shap_values, display=False))

In [ ]:
import matplotlib.pyplot as plt
from IPython.core.display import HTML
HTML(shap.plots.text(shap_values[:,:,classes[1]], display=True))
with open('/content/drive/MyDrive/HMPV Sentiment analysis/Figures/shap classwiseNeutral.html', 'w') as file:
     file.write(shap.plots.text(shap_values[:,:,classes[1]], display=False))

In [ ]:
shap.initjs
plt.figure(figsize=(9,13))
shap.plots.bar(shap_values[:,:,classes[0]].mean(0),max_display=20)
plt.savefig('/content/drive/MyDrive/HMPV Sentiment analysis/Figures/shapvaluesbar.pdf')
plt.close()

In [ ]:
shap.plots.bar(shap_values[:,:,classes[1]].mean(0),max_display=20)

In [ ]:
shap.plots.bar(shap_values[:,:,classes[2]].mean(0),max_display=20)